# 07 - CNN Explainability Heatmaps

This notebook checks what the two CNN iterations focus on for selected validation images. It uses Grad-CAM heatmaps for each model's predicted class logit. The heatmaps are used as inspection evidence: they can show whether the model mainly responds to fabric regions, but they do not prove that a prediction is correct.

The notebook compares the same nine validation images for both CNNs: three images each from the three largest validation classes.

## Setup

The setup mirrors the CNN notebooks. The code runs on CPU, uses the `fabrics` environment, and loads the existing DenseNet201 weights from the local torch cache when available.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import random
from typing import Iterable

local_torch_home = Path.home() / ".cache" / "torch"
local_densenet_weights = local_torch_home / "hub" / "checkpoints" / "densenet201-c1103571.pth"
if local_densenet_weights.exists():
    os.environ.setdefault("TORCH_HOME", str(local_torch_home))
else:
    os.environ.setdefault("TORCH_HOME", "/tmp/mdpp_v2_notebook_cache/torch")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mdpp_v2_notebook_cache/matplotlib")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.models import DenseNet201_Weights, densenet201

The constants below intentionally match the training notebooks. This prevents the explainability check from accidentally using a different image subset, class order, or checkpoint.

In [ ]:
RANDOM_SEED = 42
BATCH_SIZE = 16
IMAGE_MODALITY = "Stereo"
IMAGE_INSTRUMENT = "T"
SELECTED_SETTING = "Colour + mono + polarized grayscale"
FROZEN_CNN_CHECKPOINT_NAME = "cnn_frozen_densenet201_linear_head.pt"
FINETUNED_CNN_CHECKPOINT_NAME = "cnn_finetuned_densenet201_denseblock4_head.pt"

CLASS_NAMES = [
    "Cotton",
    "Natural polyamide",
    "Polyester / recycled polyester / PET",
    "Regenerated cellulosic",
    "Synthetic polyamide / nylon",
]
CLASS_SHORT_NAMES = {
    "Cotton": "Cotton",
    "Natural polyamide": "Natural PA",
    "Polyester / recycled polyester / PET": "Polyester/PET",
    "Regenerated cellulosic": "Regenerated cell.",
    "Synthetic polyamide / nylon": "Synthetic PA",
}
LABEL_TO_INDEX = {label: index for index, label in enumerate(CLASS_NAMES)}
INDEX_TO_LABEL = {index: label for label, index in LABEL_TO_INDEX.items()}

FEATURE_SOURCES = [
    {"feature_source": "C_gray", "formats": ("C",), "preprocessing": "L_to_RGB"},
    {"feature_source": "M_gray", "formats": ("M",), "preprocessing": "L_to_RGB"},
    {"feature_source": "P_gray", "formats": ("P",), "preprocessing": "L_to_RGB"},
]
SELECTED_FEATURE_SOURCES = tuple(source["feature_source"] for source in FEATURE_SOURCES)
SPLITS = ["val"]

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))


def select_torch_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    print("CUDA is not available; using CPU for this inference notebook.")
    return torch.device("cpu")


DEVICE = select_torch_device()
PIN_MEMORY = DEVICE.type == "cuda"
DEVICE

## 1. Paths and Validation Split

The validation split provides the fabrics used for model inspection.

In [ ]:
def resolve_mdpp_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "mdpp_v2" / "data_splits").exists():
            return candidate / "mdpp_v2"
        if candidate.name == "mdpp_v2" and (candidate / "data_splits").exists():
            return candidate
    raise FileNotFoundError("Could not resolve mdpp_v2 root from current working directory.")


MDPP_ROOT = resolve_mdpp_root()
DATA_SPLITS_DIR = MDPP_ROOT / "data_splits"
ARTIFACTS_DIR = MDPP_ROOT / "artifacts"
FROZEN_CNN_CHECKPOINT_PATH = ARTIFACTS_DIR / FROZEN_CNN_CHECKPOINT_NAME
FINETUNED_CNN_CHECKPOINT_PATH = ARTIFACTS_DIR / FINETUNED_CNN_CHECKPOINT_NAME
HEATMAP_FIGURE_PATH = ARTIFACTS_DIR / "cnn_explainability_heatmaps.png"
THESIS_HEATMAP_FIGURE_PATH = Path(os.environ.get("THESIS_HEATMAP_FIGURE_PATH", ARTIFACTS_DIR / "cnn_explainability_heatmaps.png"))

split_sample_tables = {split: pd.read_csv(DATA_SPLITS_DIR / split / "samples.csv") for split in SPLITS}
split_manifest_tables = {split: pd.read_csv(DATA_SPLITS_DIR / split / "measurements_manifest.csv") for split in SPLITS}

print(f"Using device: {DEVICE}")
print(f"Torch home: {os.environ['TORCH_HOME']}")

## 2. Build the Validation Image Table

The validation image table applies the same CNN eligibility filter as the training notebooks. `EXCLUDED_CNN` fabrics are removed before any prediction or heatmap is calculated.

In [ ]:
def eligible_samples_for_split(split: str) -> pd.DataFrame:
    samples = split_sample_tables[split].copy()
    eligible = samples[samples["cnn_label"].ne("EXCLUDED_CNN")].copy()
    eligible = eligible[["Sample ID", "classification_label", "cnn_label", "split"]]
    eligible = eligible.rename(columns={"Sample ID": "sample_id"})
    eligible["label_index"] = eligible["classification_label"].map(LABEL_TO_INDEX)
    if eligible["label_index"].isna().any():
        missing = sorted(eligible.loc[eligible["label_index"].isna(), "classification_label"].dropna().unique())
        raise ValueError(f"Unexpected retained CNN labels: {missing}")
    return eligible


def build_source_table(split: str, feature_source: str, formats: Iterable[str], preprocessing: str) -> pd.DataFrame:
    manifest = split_manifest_tables[split].copy()
    eligible = eligible_samples_for_split(split)
    mask = (
        manifest["modality_group"].eq("image")
        & manifest["modality"].eq(IMAGE_MODALITY)
        & manifest["instrument"].eq(IMAGE_INSTRUMENT)
        & manifest["data_format"].isin(tuple(formats))
    )
    table = manifest.loc[mask].merge(eligible, on="sample_id", how="inner", suffixes=("", "_sample"))
    table["path"] = table["final_path"].map(Path)
    table["image_path"] = table["path"]
    table["feature_source"] = feature_source
    table["preprocessing"] = preprocessing
    table["setting"] = SELECTED_SETTING
    table["source_split"] = split
    table["image_id"] = table["source_split"] + ":" + table["feature_source"] + ":" + table["filename"]
    if table["cnn_label"].eq("EXCLUDED_CNN").any():
        raise ValueError("EXCLUDED_CNN rows entered the explainability table.")
    missing_files = [path for path in table["path"] if not path.exists()]
    if missing_files:
        raise FileNotFoundError(f"Missing image files, first examples: {missing_files[:3]}")
    return table.sort_values(["sample_id", "feature_source", "replicate", "filename"]).reset_index(drop=True)

The selected image representation is grayscale `C+M+P`. Each file is still passed to DenseNet201 as three channels later, because ImageNet-pretrained DenseNet201 expects RGB-shaped input.

In [ ]:
source_tables = []
for source_spec in FEATURE_SOURCES:
    source_tables.append(
        build_source_table(
            split="val",
            feature_source=source_spec["feature_source"],
            formats=source_spec["formats"],
            preprocessing=source_spec["preprocessing"],
        )
    )

val_table = (
    pd.concat(source_tables, ignore_index=True)
    .sort_values(["sample_id", "feature_source", "replicate", "filename"])
    .reset_index(drop=True)
)
val_table["row_index"] = np.arange(len(val_table))

validation_input_summary = pd.DataFrame([
    {
        "split": "validation",
        "files": len(val_table),
        "fabrics": val_table["sample_id"].nunique(),
        "image_formats": "+".join(sorted(val_table["data_format"].unique())),
        "preprocessing": "L -> RGB grayscale",
    }
])
validation_input_summary

## 3. CNN Models

Both models are loaded from their saved checkpoints. Parameters stay frozen for prediction, but Grad-CAM will still need gradients through the forward pass later to calculate the heatmap.

In [ ]:
weights = DenseNet201_Weights.DEFAULT
weights_transform = weights.transforms()
NORMALIZE_MEAN = np.array(weights_transform.mean, dtype="float32").reshape(3, 1, 1)
NORMALIZE_STD = np.array(weights_transform.std, dtype="float32").reshape(3, 1, 1)


class DenseNetClassifier(nn.Module):
    def __init__(self, features: nn.Module, classifier: nn.Linear):
        super().__init__()
        self.features = features
        self.classifier = classifier

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        features = self.features(images)
        out = F.relu(features, inplace=False)
        out = F.adaptive_avg_pool2d(out, (1, 1))
        embeddings = torch.flatten(out, 1)
        return self.classifier(embeddings)


def strip_state_dict_prefix(state_dict: dict[str, torch.Tensor], prefix: str) -> dict[str, torch.Tensor]:
    return {key.removeprefix(prefix): value for key, value in state_dict.items() if key.startswith(prefix)}


def freeze_for_explainability(model: nn.Module) -> nn.Module:
    for parameter in model.parameters():
        parameter.requires_grad = False
    return model.to(DEVICE).eval()

The two loader functions reconstruct the exact model paths used in the training notebooks. The frozen model loads only the linear head; the fine-tuned model loads the updated final DenseNet block and head.

In [ ]:
def load_frozen_iteration_model() -> tuple[DenseNetClassifier, dict]:
    checkpoint = torch.load(FROZEN_CNN_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    if checkpoint.get("class_names") != CLASS_NAMES:
        raise ValueError("Frozen CNN checkpoint class order does not match this notebook.")

    base = densenet201(weights=weights)
    feature_dim = base.classifier.in_features
    base.classifier = nn.Linear(feature_dim, len(CLASS_NAMES))
    base.classifier.load_state_dict(checkpoint["state_dict"])
    model = DenseNetClassifier(base.features, base.classifier)
    return freeze_for_explainability(model), checkpoint


def load_finetuned_iteration_model() -> tuple[DenseNetClassifier, dict]:
    checkpoint = torch.load(FINETUNED_CNN_CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    if checkpoint.get("class_names") != CLASS_NAMES:
        raise ValueError("Fine-tuned CNN checkpoint class order does not match this notebook.")

    base = densenet201(weights=weights)
    feature_dim = base.classifier.in_features
    base.classifier = nn.Linear(feature_dim, len(CLASS_NAMES))
    tail_state = checkpoint["tail_state_dict"]
    base.features.denseblock4.load_state_dict(strip_state_dict_prefix(tail_state, "denseblock4."))
    base.features.norm5.load_state_dict(strip_state_dict_prefix(tail_state, "norm5."))
    base.classifier.load_state_dict(strip_state_dict_prefix(tail_state, "classifier."))
    model = DenseNetClassifier(base.features, base.classifier)
    return freeze_for_explainability(model), checkpoint

The checkpoint table is shown so it is clear that the notebook loaded saved models instead of training new ones.

In [ ]:
iteration1_model, iteration1_checkpoint = load_frozen_iteration_model()
iteration2_model, iteration2_checkpoint = load_finetuned_iteration_model()

checkpoint_status = pd.DataFrame([
    {
        "model": "iteration_1_frozen_densenet201_linear_head",
        "checkpoint": str(FROZEN_CNN_CHECKPOINT_PATH),
        "status": "loaded",
        "best_epoch": int(iteration1_checkpoint["best_epoch"]),
    },
    {
        "model": "iteration_2_finetuned_denseblock4",
        "checkpoint": str(FINETUNED_CNN_CHECKPOINT_PATH),
        "status": "loaded",
        "best_epoch": int(iteration2_checkpoint["best_epoch"]),
    },
])
checkpoint_status

## 4. Predict Validation Images

The model predictions are needed before selecting examples and before choosing each Grad-CAM target class. The heatmap target is the class predicted by the model, because the question is what supported that model output.

In [ ]:
def preprocess_image_from_row(row: pd.Series) -> torch.Tensor:
    with Image.open(row["path"]) as image:
        # DenseNet201 expects three channels. The image content remains grayscale.
        if row["preprocessing"] == "L_to_RGB":
            image = image.convert("L").convert("RGB")
        elif row["preprocessing"] == "RGB":
            image = image.convert("RGB")
        else:
            raise ValueError(f"Unknown preprocessing mode: {row['preprocessing']}")
        return weights_transform(image)


class InferenceImageDataset(Dataset):
    def __init__(self, table: pd.DataFrame):
        self.table = table.reset_index(drop=True).copy()

    def __len__(self) -> int:
        return len(self.table)

    def __getitem__(self, idx: int):
        row = self.table.iloc[idx]
        return preprocess_image_from_row(row), int(row["row_index"])

Image-level predictions are calculated first. These predictions provide both the class labels and confidence values used for example selection.

In [ ]:
@torch.no_grad()
def predict_image_rows(model: nn.Module, table: pd.DataFrame, model_name: str) -> pd.DataFrame:
    dataset = InferenceImageDataset(table)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=PIN_MEMORY)
    rows = []
    metadata = table.set_index("row_index")

    for images, row_indices in loader:
        logits = model(images.to(DEVICE, non_blocking=PIN_MEMORY)).cpu()
        probabilities = torch.softmax(logits, dim=1).numpy()
        for row_index, probability in zip(row_indices.numpy(), probabilities):
            source = metadata.loc[int(row_index)]
            pred_index = int(probability.argmax())
            row = {
                "model": model_name,
                "row_index": int(row_index),
                "sample_id": source["sample_id"],
                "filename": source["filename"],
                "feature_source": source["feature_source"],
                "data_format": source["data_format"],
                "true_label": source["classification_label"],
                "true_index": int(source["label_index"]),
                "pred_index": pred_index,
                "pred_label": INDEX_TO_LABEL[pred_index],
                "confidence": float(probability[pred_index]),
            }
            row.update({label: float(probability[index]) for index, label in enumerate(CLASS_NAMES)})
            rows.append(row)
    return pd.DataFrame(rows)

Fabric-level predictions are still calculated as a compact validation summary. The image selection and heatmaps are image-level, because each Grad-CAM overlay explains one image prediction.

In [ ]:
def aggregate_fabric_predictions(image_predictions: pd.DataFrame, model_name: str) -> pd.DataFrame:
    rows = []
    for sample_id, group in image_predictions.groupby("sample_id", sort=True):
        mean_probs = group[CLASS_NAMES].mean()
        pred_label = mean_probs.idxmax()
        true_labels = group["true_label"].unique()
        if len(true_labels) != 1:
            raise ValueError(f"Multiple true labels found for {sample_id}: {true_labels}")
        rows.append({
            "model": model_name,
            "sample_id": sample_id,
            "true_label": true_labels[0],
            "pred_label": pred_label,
            "pred_index": LABEL_TO_INDEX[pred_label],
            "confidence": float(mean_probs[pred_label]),
            "correct": bool(pred_label == true_labels[0]),
            "image_count": len(group),
        })
    return pd.DataFrame(rows)


iteration1_image_predictions = predict_image_rows(iteration1_model, val_table, "iteration_1")
iteration2_image_predictions = predict_image_rows(iteration2_model, val_table, "iteration_2")
iteration2_fabric_predictions = aggregate_fabric_predictions(iteration2_image_predictions, "iteration_2")

print(f"Validation images predicted: {len(iteration2_image_predictions)}")
print(f"Validation fabrics predicted: {len(iteration2_fabric_predictions)}")

## 5. Select Top-Class Images

The heatmap figure compares the three largest validation classes. This makes the output easier to read without textile expertise: each class gets the same number of image examples, and both CNNs are shown on the same images.

The selection unit is the validation image file. The smaller retained classes are not used in this figure because they have too few validation examples for a stable visual comparison.

In [ ]:
def top_validation_classes(n_classes: int = 3) -> list[str]:
    counts = val_table["classification_label"].value_counts()
    top_classes = counts.head(n_classes).index.tolist()
    if len(top_classes) != n_classes:
        raise ValueError(f"Expected {n_classes} classes, got {len(top_classes)}")
    return top_classes


def confidence_tier_positions(n_items: int) -> dict[str, int]:
    if n_items < 3:
        raise ValueError(f"Need at least three candidate images, got {n_items}")
    return {
        "low confidence": 0,
        "median confidence": n_items // 2,
        "high confidence": n_items - 1,
    }


def add_tier_selection(selected_rows: list[dict], candidates: pd.DataFrame, class_label: str, tier: str, position: int) -> None:
    image_row = candidates.iloc[position]
    selected_rows.append({
        "row_index": int(image_row["row_index"]),
        "class_group": class_label,
        "confidence_tier": tier,
        "selection_reason": f"{CLASS_SHORT_NAMES.get(class_label, class_label)}: {tier}",
    })

The selection rule is deterministic. Within each class, the notebook first tries to use images that iteration 2 predicts correctly. It then selects low, median, and high confidence examples. If there are fewer than three correct images for a class, it falls back to all true-class images ranked by the model probability for the true class.

In [ ]:
def candidate_images_for_class(class_label: str) -> pd.DataFrame:
    class_rows = iteration2_image_predictions[iteration2_image_predictions["true_label"].eq(class_label)].copy()
    correct_rows = class_rows[class_rows["pred_label"].eq(class_rows["true_label"])].copy()

    if len(correct_rows) >= 3:
        candidates = correct_rows.assign(selection_pool="correct prediction")
        score_column = "confidence"
    else:
        candidates = class_rows.assign(
            selection_pool="all true-class images",
            true_class_score=class_rows[class_label],
        )
        score_column = "true_class_score"

    candidates = candidates.sort_values([score_column, "sample_id", "filename"], ascending=[True, True, True])
    if len(candidates) < 3:
        raise ValueError(f"Class {class_label} has fewer than three candidate validation images.")
    return candidates.reset_index(drop=True)


def select_explainability_images() -> pd.DataFrame:
    selected_rows: list[dict] = []
    for class_label in top_validation_classes(3):
        candidates = candidate_images_for_class(class_label)
        for tier, position in confidence_tier_positions(len(candidates)).items():
            add_tier_selection(selected_rows, candidates, class_label, tier, position)

    selected = pd.DataFrame(selected_rows)
    if len(selected) != 9:
        raise ValueError(f"Expected 9 selected validation images, got {len(selected)}")
    return selected

The selected-image table is the main index for the figure: it records the class group, confidence tier, source image, and what both CNNs predicted.

In [ ]:
selected_rows = select_explainability_images()
selected_base = selected_rows.merge(
    val_table[["row_index", "sample_id", "filename", "feature_source", "data_format", "classification_label", "path", "preprocessing", "label_index"]],
    on="row_index",
    how="left",
)
selected_image_table = (
    selected_base
    .merge(
        iteration1_image_predictions[["row_index", "pred_label", "confidence"]].rename(
            columns={"pred_label": "iteration_1_pred", "confidence": "iteration_1_confidence"}
        ),
        on="row_index",
        how="left",
    )
    .merge(
        iteration2_image_predictions[["row_index", "pred_label", "confidence"]].rename(
            columns={"pred_label": "iteration_2_pred", "confidence": "iteration_2_confidence"}
        ),
        on="row_index",
        how="left",
    )
)
selected_image_table["true_label"] = selected_image_table["classification_label"]
selected_image_table = selected_image_table[
    ["row_index", "class_group", "confidence_tier", "selection_reason", "sample_id", "filename", "data_format", "true_label",
     "iteration_1_pred", "iteration_1_confidence", "iteration_2_pred", "iteration_2_confidence"]
]
selected_image_table.drop(columns=["row_index"])

## 6. Calculate Grad-CAM Heatmaps

Grad-CAM uses the final convolutional feature map and the gradient of the selected class logit. Here the selected class is each model's own prediction for the image. The implementation keeps model weights frozen; gradients are used only to calculate the heatmap.

In [ ]:
class GradCAM:
    def __init__(self, model: DenseNetClassifier, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations: torch.Tensor | None = None
        self.gradients: torch.Tensor | None = None
        self.forward_handle = target_layer.register_forward_hook(self._store_activations)
        self.backward_handle = target_layer.register_full_backward_hook(self._store_gradients)

    def _store_activations(self, module: nn.Module, inputs: tuple[torch.Tensor, ...], output: torch.Tensor) -> None:
        # Forward hooks capture the feature map used by Grad-CAM.
        self.activations = output

    def _store_gradients(self, module: nn.Module, grad_input: tuple[torch.Tensor, ...], grad_output: tuple[torch.Tensor, ...]) -> None:
        # Backward hooks capture how strongly each feature map affects the selected class logit.
        self.gradients = grad_output[0]

    def remove(self) -> None:
        self.forward_handle.remove()
        self.backward_handle.remove()

    def heatmap(self, image_tensor: torch.Tensor, target_index: int) -> np.ndarray:
        self.model.zero_grad(set_to_none=True)
        self.activations = None
        self.gradients = None
        batch = image_tensor.unsqueeze(0).to(DEVICE)
        batch.requires_grad_(True)
        score = self.model(batch)[:, target_index].sum()
        score.backward()
        if self.activations is None or self.gradients is None:
            raise RuntimeError("Grad-CAM hooks did not capture activations and gradients.")
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=batch.shape[-2:], mode="bilinear", align_corners=False)
        heatmap = cam[0, 0].detach().cpu().numpy()
        heatmap_min = float(heatmap.min())
        heatmap_max = float(heatmap.max())
        if heatmap_max <= heatmap_min:
            return np.zeros_like(heatmap, dtype="float32")
        return ((heatmap - heatmap_min) / (heatmap_max - heatmap_min)).astype("float32")

The display helpers reverse DenseNet normalisation and overlay the heatmap on the preprocessed input image.

In [ ]:
def denormalize_tensor(image_tensor: torch.Tensor) -> np.ndarray:
    array = image_tensor.detach().cpu().numpy()
    array = (array * NORMALIZE_STD) + NORMALIZE_MEAN
    array = np.clip(array, 0.0, 1.0)
    return np.transpose(array, (1, 2, 0))


def overlay_heatmap(base_image: np.ndarray, heatmap: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    heatmap_rgb = plt.get_cmap("hot")(heatmap)[..., :3]
    return np.clip((1.0 - alpha) * base_image + alpha * heatmap_rgb, 0.0, 1.0)


def class_title(label: str) -> str:
    return CLASS_SHORT_NAMES.get(label, label)

The heatmap generation loop applies Grad-CAM to both CNNs for each selected image, using each model’s own predicted class.

In [ ]:
def heatmaps_for_selected_images() -> tuple[pd.DataFrame, dict[tuple[int, str], np.ndarray], dict[int, np.ndarray]]:
    iteration1_cam = GradCAM(iteration1_model, iteration1_model.features.norm5)
    iteration2_cam = GradCAM(iteration2_model, iteration2_model.features.norm5)
    details = []
    heatmaps: dict[tuple[int, str], np.ndarray] = {}
    base_images: dict[int, np.ndarray] = {}

    try:
        prediction_lookup = {
            "iteration_1": iteration1_image_predictions.set_index("row_index"),
            "iteration_2": iteration2_image_predictions.set_index("row_index"),
        }
        for _, table_row in selected_base.iterrows():
            row_index = int(table_row["row_index"])
            image_tensor = preprocess_image_from_row(table_row)
            base_images[row_index] = denormalize_tensor(image_tensor)

            for model_key, cam_runner in [("iteration_1", iteration1_cam), ("iteration_2", iteration2_cam)]:
                pred_row = prediction_lookup[model_key].loc[row_index]
                heatmap = cam_runner.heatmap(image_tensor, int(pred_row["pred_index"]))
                heatmaps[(row_index, model_key)] = heatmap
                details.append({
                    "row_index": row_index,
                    "model": model_key,
                    "target_label": pred_row["pred_label"],
                    "heatmap_min": float(heatmap.min()),
                    "heatmap_max": float(heatmap.max()),
                    "heatmap_std": float(heatmap.std()),
                    "heatmap_finite": bool(np.isfinite(heatmap).all()),
                })
    finally:
        iteration1_cam.remove()
        iteration2_cam.remove()
    return pd.DataFrame(details), heatmaps, base_images

This cell generates the Grad-CAM heatmaps and shows a compact summary of the generated heatmap values.

In [ ]:
heatmap_details, heatmap_arrays, base_image_arrays = heatmaps_for_selected_images()

heatmap_quality_summary = pd.DataFrame([
    {
        "heatmaps": len(heatmap_details),
        "images": selected_image_table[["sample_id", "filename"]].drop_duplicates().shape[0],
        "min_value": heatmap_details["heatmap_min"].min(),
        "max_value": heatmap_details["heatmap_max"].max(),
        "mean_std": heatmap_details["heatmap_std"].mean(),
        "all_finite": bool(heatmap_details["heatmap_finite"].all()),
    }
])
heatmap_quality_summary

## 7. Heatmap Figure

Each row shows one validation image. The figure is grouped by the three largest validation classes, with low, median, and high confidence examples for each class. The first column is the DenseNet input after preprocessing. The second and third columns show Grad-CAM overlays for iteration 1 and iteration 2.

In [ ]:
fig, axes = plt.subplots(9, 3, figsize=(11, 25))
selected_plot_rows = selected_image_table.set_index("row_index")

for row_position, row in enumerate(selected_base.itertuples(index=False)):
    row_index = int(row.row_index)
    table_row = selected_plot_rows.loc[row_index]
    base_image = base_image_arrays[row_index]
    iteration1_overlay = overlay_heatmap(base_image, heatmap_arrays[(row_index, "iteration_1")])
    iteration2_overlay = overlay_heatmap(base_image, heatmap_arrays[(row_index, "iteration_2")])
    class_group = class_title(table_row["class_group"])
    tier = table_row["confidence_tier"]

    axes[row_position, 0].imshow(base_image)
    axes[row_position, 0].set_title(
        f"{class_group}: {tier}\n{row.sample_id} ({row.data_format})",
        fontsize=9,
    )
    axes[row_position, 1].imshow(iteration1_overlay)
    axes[row_position, 1].set_title(
        f"Iteration 1\npred: {class_title(table_row['iteration_1_pred'])}, conf={table_row['iteration_1_confidence']:.2f}",
        fontsize=9,
    )
    axes[row_position, 2].imshow(iteration2_overlay)
    axes[row_position, 2].set_title(
        f"Iteration 2\npred: {class_title(table_row['iteration_2_pred'])}, conf={table_row['iteration_2_confidence']:.2f}",
        fontsize=9,
    )
    for axis in axes[row_position]:
        axis.axis("off")

plt.tight_layout()
fig.savefig(HEATMAP_FIGURE_PATH, dpi=200, bbox_inches="tight")
fig.savefig(THESIS_HEATMAP_FIGURE_PATH, dpi=200, bbox_inches="tight")
plt.show()

## 8. Interpretation Note

The figure functions as a class-comparison view. The practical question is whether the two CNN iterations respond to fabric-surface texture regions, and whether their focus appears to differ between Cotton, Polyester/PET, and Synthetic polyamide/nylon.

The heatmaps show which image regions influenced the selected class logit for these examples.